In [1]:
from openai import OpenAI
import pandas as pd
import tiktoken
import csv
import lxml

from typing import List
from pydantic import BaseModel

In [4]:
#Read dataset descriptions for Review

data_source_tab= pd.read_csv('all_icpsr_metadata.csv', header = 0)
data_source_tab=data_source_tab.drop(range(5,14))
data_source_tab['Keywords'] = data_source_tab['Keywords'].str.replace('|', ',', regex=False)
data_source_tab.columns = ["Name", "Desc", "ICPSR_Tags"]

print(len(data_source_tab))

12647


In [39]:
#Remove datasets withou descriptions

data_source_tab = data_source_tab.dropna(how='any', subset=['Name', 'Desc'])
print(len(data_source_tab))

4319


In [40]:
#Select a random sample 500 records with description

data_source_tab_sampled = data_source_tab.sample(n=500, random_state=42)
data_source_tab_sampled = data_source_tab_sampled.reset_index(drop=True)

print(data_source_tab_sampled.head())

                                                Name  \
0  Longitudinal Evaluation of Chicago's Community...   
1         National Mortality Followback Survey, 1993   
2  Uniform Crime Reporting Program Data [United S...   
3  Enhanced Services for the Hard-to-Employ Cente...   
4  Arab Barometer: Public Opinion Survey Conducte...   

                                                Desc  \
0  The purpose of this study was to evaluate the ...   
1  The National Mortality Followback Survey (NMFS...   
2  These data provide information on the number o...   
3  The Enhanced Services for the Hard-to-Employ (...   
4  The Arab Barometer is a multicountry social su...   

                                          ICPSR_Tags  
0  citizen participation  |  community involvemen...  
1  accidents  |  causes of death  |  death record...  
2  arrest records  |  arrests  |  crime rates  | ...  
3  child support  |  conviction records  |  crimi...  
4  Arab Israeli conflict  |  democracy  |  econom..

In [6]:
#Get ChatGPT API key

gpt_api=pd.read_csv('chat_gpt_key.csv')['key'].values[0]
client = OpenAI(api_key=gpt_api)

In [7]:
#Read ICPSR Thesaurus

ICPSR = pd.read_xml('subject_terms.xml')

ICPSR_list=ICPSR['DESCRIPTOR']
ICPSR_list=list(set(ICPSR_list))

In [8]:
ICPSR_list[0:10]

['statistical models',
 'tax penalties',
 'livestock value',
 'environmental monitoring',
 'reading ability',
 'Americans for Democratic Action',
 'mental disorders',
 'political history',
 'World Trade Organization',
 'drugs']

In [9]:
len(ICPSR_list)

3282

In [19]:
#Create GPT Function

def gen_terms(dataset_name, thesaurus, temp = 1, freq_pen = 0 ): 
    review=data_source_tab_sampled[data_source_tab_sampled['Name'] == dataset_name]
    review=str(review['Name'] + " " + review['Desc'])
    review=str(review).replace('\\','')
    
    system_prompt = f"""
    You are reviewing abstracts and need to select terms that best describe the data. 
    Use the description provided and your own knowledge of the dataset, select only the appropriate terms 
    from the following list: {thesaurus}.
    Instructions:
    1. Select the most relevant terms based on the abstract and your own knowledge of the dataset.
    2. Return the terms in a comma separated list with no other punctuation, sorted by descending relevance.
    3. Include only the top 5 terms.
    Please ensure that no other terms outside these lists are included in your response.
    """
   
    response=client.chat.completions.create(
         messages=[
             {'role': 'system',
              'content': system_prompt
             },
             {
                 "role": "user",
                 "content": review,    
             }  
         ],
         model="gpt-4o",
         temperature=temp,
         frequency_penalty=freq_pen
     )
    
    value_IC=response.choices[0].message.content
    
    return value_IC

In [13]:
#Create function to remove hallucinated terms

def check_terms(response_string, thesaurus):
    
    lower_thesaurus = list(pd.Series(thesaurus).str.lower())
    lower_resp      = response_string.lower()
    
    response_array = lower_resp.split(",")
    
    real_terms = []
    fake_terms = []
    
    for i in response_array:
        test_term = i.lstrip(' ')
        if test_term in lower_thesaurus:
            real_terms.append(test_term)
        else:
            fake_terms.append(test_term)
   
    if len(real_terms) == 0: 
        real_terms = 'NA'
    
    return(real_terms)

In [41]:
#Run Chatgpt api call and remove halluncated terms

icpsr_results = []
icpsr_real = []


for i in range(len(data_source_tab_sampled)):
    
    dataset_name  = data_source_tab_sampled.at[i, 'Name']
    
    icpsr_one_result = str(gen_terms(dataset_name, ICPSR_list))

    icpsr_one_real = check_terms(icpsr_one_result, ICPSR_list)
    icpsr_one_real = ', '.join(icpsr_one_real)
    
    icpsr_results.append(icpsr_one_result)

    icpsr_real.append(icpsr_one_real)


data_source_tab_sampled['Model Selected Terms']=icpsr_results

data_source_tab_sampled['Model Selected Terms - No Hallucinations']=icpsr_real


In [42]:
output = data_source_tab_sampled

In [43]:
#Rename columns and change term seperator to |

columns_to_replace = ['ICPSR_Tags', 'Model Selected Terms', 'Model Selected Terms - No Hallucinations']
for col in columns_to_replace:
   output[col] = output[col].str.replace(',', ' | ', regex=False)

In [44]:
#clean text

columns_to_replace = ['Name', 'Desc']
for col in columns_to_replace:
    output[col] = output[col].str.replace(',', '', regex=False)

In [45]:
#save to csv
output.to_csv("output_ICPSR_500.csv", index=False)

In [13]:
print(output.head())

,Name,Desc,ICPSR_Tags,Model Selected Terms,Model Selected Terms - No Hallucinations
0,Changes in the Distribution of Wealth: Increas...,The data collection describes changes in the d...,economic conditions | households | income dis...,wealth | economic inequality | economic cond...,wealth | economic conditions | income distri...
1,Midlife in the United States: Core Sample Mort...,In 1995-1996 the MacArthur Midlife Research Ne...,adults | aging | death | death records | mor...,aging | midlife | adult education | health ...,aging | midlife | adult education | health ...
2,Improving Family-Centered Pediatric Trauma Car...,The current standard of care in the treatment ...,children | emergency departments | health care...,families | trauma centers | child health | ...,families | trauma centers | child health | ...
3,A Blue Start: A large-scale pairwise and highe...,This dataset consists of all starter packs and...,social media |social networks,computer programs | technology | innovation ...,computer programs | technology | innovation ...
4,Shelby County School District Comprehensive Sc...,This study used a school-randomized design to ...,intervention strategies | juveniles | middle...,educational facilities | education | school ...,educational facilities | education | school ...
